In [1]:
import asyncio, aiohttp, re, json, time, pandas as pd
from bs4 import BeautifulSoup
from unidecode import unidecode
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from tqdm.asyncio import tqdm_asyncio
from tqdm import tqdm

In [2]:
def make_driver(headless: bool = True) -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    return webdriver.Chrome(service=webdriver.chrome.service.Service(
                                ChromeDriverManager().install()),
                            options=options)

In [3]:
START_URL = "https://food.ru"


In [4]:
def infinite_scroll(url: str, target_count: int = 300, pause: float = 0.5, scroll_ratio=0.7) -> list[dict]:
    """Прокручиваем страницу, пока не соберём target_count карточек."""
    seen = set()
    records = []
    driver = make_driver()

    for page in range(1, 100):
        page_url = url if page == 1 else f"{url}?page={page}"
        driver.get(page_url)
            
        # ищем все listitem-карточки в секции "Лента публикаций"
        cards = driver.find_elements(
            By.CSS_SELECTOR,
            'section[aria-label="Лента публикаций"] div[role="listitem"] a.card_card__YG0I9'
        )
        print(f"Url {page_url}, Found {len(cards)} cards, total: {len(records)}")
        
        if not cards:
            print(f"Url {page_url}, No cards found, stopping.")
            break
            
        for a in cards:
            href = a.get_attribute("href")
            if href and href not in seen:
                img = a.find_element(By.CSS_SELECTOR, "img")
                title = img.get_attribute("alt") or img.get_attribute("title")
                print(f"Title: {title}, URL: {href}")
                records.append({"url": href, "title": title})
                seen.add(href)
        
        if len(records) >= target_count:
            print(f"Reached target count: {target_count} recipes.")
            break
            
        # прокручиваем дальше
        total_height = driver.execute_script("return document.body.scrollHeight")
        driver.execute_script("window.scrollTo(0, arguments[0]);", total_height * scroll_ratio)
        time.sleep(pause)
        
    driver.quit()
    return records

In [5]:
HEADERS = {"User-Agent": "Mozilla/5.0"}
RE_NUM  = re.compile(r"\b\d+[.,]?\d*\b")
RE_PAREN= re.compile(r"\([^)]*\)")

def clean_name(s: str) -> str:
    s = unidecode(s)
    s = RE_PAREN.sub("", s)
    s = RE_NUM.sub("", s)
    return s.lower().strip()

In [6]:
from aiohttp import ClientTimeout

timeout = ClientTimeout(
    total=120,        # общий таймаут (можно больше)
    connect=20,      # на соединение
    sock_read=20     # на чтение ответа
)

In [16]:
async def parse_recipe(session: aiohttp.ClientSession, url: str, title: str, sem: asyncio.Semaphore) -> dict:
    recipe_data = {"title": title, "url": url}
    async with sem:
        try:
            await asyncio.sleep(0.5)
            async with session.get(url, headers=HEADERS, timeout=timeout) as r:
                html = await r.text()
        except (asyncio.TimeoutError, aiohttp.ClientError, RuntimeError) as e:
            print('Exception: ', e)
            recipe_data.update({
                "recipe": "Ошибка",
                "time": 0,
                "count": 0,
                "ingredients": [],
                "nutrients": {},
                "allergy": "Ошибка"
            })
            return recipe_data
        
        soup = BeautifulSoup(html, "lxml")
        
        # рецепт
        section = soup.find('section', id='step-by-step-recipe')
        if section:
            spans = section.find_all('span', class_='markup_text__F9WKe')
    
            if spans and len(spans) > 1:
                # Пропускаем первый элемент и собираем текст из оставшихся
                combined_text = '\n'.join([span.text.strip() for span in spans[1:]])
                recipe_data['recipe'] = combined_text.strip()  # Убираем лишние пробелы
            else:
                recipe_data['recipe'] = "Описание не найдено"
        else:
            recipe_data['recipe'] = "Секция с рецептом не найдена"
        
        # время
        ready_time = soup.find('meta', itemprop='totalTime')
        ready_minutes = ready_time['content'] if ready_time else 'PT0M'
        ready_minutes_value = int(ready_minutes.replace('PT', '').replace('M', '')) if ready_minutes else 0
        recipe_data['time'] = ready_minutes_value
        
        # количество порций
        servings_input = soup.find('input', class_='input yield default yield')
        if servings_input:
            servings_value = servings_input.get('value')        
            if servings_value.isdigit():
                servings_value = int(servings_value)
                recipe_data['count'] = servings_value
            else:
                recipe_data['count'] = 1
            
        
        # ингридиенты
        ing_rows = []
        for tr in soup.find_all("tr", {"itemprop": "recipeIngredient"}):
            name_tag = tr.find("span", class_="name")
            value_tag= tr.find("span", class_="value")  # число в граммах
    
            # qty + ед. измерения в исходном правом столбце:
            #   '4 ... шт. = 240 г'
            qty_block = tr.find("span", class_="ingredientsTable_text__3ILFA")
            qty_text  = qty_block.get_text(" ", strip=True) if qty_block else ""
            qty_match = re.search(r"^([\d,\.]+)", qty_text)  # первые цифры
    
            ing_rows.append({
                "name":       name_tag.get_text(strip=True) if name_tag else None,
                "qty":        qty_match.group(1).replace(",", ".") if qty_match else None,
                "grams":      value_tag.get_text(strip=True) if value_tag else None,
            })
        recipe_data['ingredients'] = ing_rows
        
        # БЖУ
        nutrient_info = {}
        nutrients = soup.find_all('span', class_='nutrient_title__JDSmX')
        values = soup.find_all('span', class_='nutrient_value__dd48k')
        if len(nutrients) == len(values):
            for nutrient, value in zip(nutrients, values):
                nutrient_info[nutrient.text.strip()] = value.text.strip()
        recipe_data['nutrients'] = nutrient_info
        
        # аллергены
        properties = soup.find_all('div', class_='properties_value__kAeD9')
        if properties:
            last_property = properties[-1].text.strip()  # Получаем последний элемент
            recipe_data['allergy'] = last_property
        else:
            recipe_data['allergy'] = "Аллергии не найдены"
        
        return recipe_data

In [17]:
# Категории, ссылки и количество страниц для каждой категории
categories = ['первые блюда', 'вторые блюда', 'закуски', 'салаты', 'гарниры', 'десерты', 'выпечка', 'напитки']
category_links = ['/recipes/pervye-bliuda', '/recipes/vtorye-bliuda', '/recipes/zakuski', '/recipes/salaty', '/recipes/garniry', '/recipes/deserty', '/recipes/vypechka', '/recipes/napitki']
category_numbers = ['384', '2631', '775', '804', '90', '820', '700', '277']  # Количество страниц
category_numbers_small = ['384', '500', '775', '804', '90', '820', '700', '277']  # Количество страниц


In [19]:
async def crawl_async(max_conn: int = 20):
    df = pd.DataFrame()
    sem = asyncio.Semaphore(max_conn)

    async with aiohttp.ClientSession() as sess:
        for cnt, url, cat in zip(category_numbers, category_links, categories):
            if cat != 'вторые блюда':
                continue  # Пропускаем все кроме вторых блюд
            cur_url = START_URL + url
            cards = infinite_scroll(cur_url, target_count=int(cnt))  # получаем карточки рецептов
            
            tasks = [parse_recipe(sess, card["url"], card["title"], sem) for card in cards]
            
            results = await asyncio.gather(*tasks, return_exceptions=True)

            recipes = []
            for recipe in results:
                if isinstance(recipe, Exception):
                    continue
                recipe['category'] = cat
                recipes.append(recipe)
            
            df_cur = pd.DataFrame(recipes)
            df_cur.to_csv(f"recipes_foodru_{cat}.csv", encoding="utf-8")
            df = pd.concat([df, df_cur], ignore_index=True)

    return df

In [21]:
import nest_asyncio 
nest_asyncio.apply()
df = asyncio.run(crawl_async(max_conn=20))

Url https://food.ru/recipes/vtorye-bliuda, Found 40 cards, total: 0
Title: Гнезда с тефтелями в томатном соусе, URL: https://food.ru/recipes/257868-gnezda-s-tefteljami-v-tomatnom-souse
Title: Тушеные баклажаны с фасолью, URL: https://food.ru/recipes/257915-baklazhany-s-fasoliu-tushenye
Title: Cалат из зеленого горошка с сырокопченым карпаччо, URL: https://food.ru/recipes/258025-calat-iz-zelenogo-goroshka-i-karpachcho-po-domashnemu-iz-indeiki-syrokopchenoi
Title: Легкий салат с кукурузой, URL: https://food.ru/recipes/255758-legkii-salat-s-kukuruzoi
Title: Фетучини карбонара, URL: https://food.ru/recipes/257663-fetuchini-karbonara
Title: Салат «Ацецили» с курицей, URL: https://food.ru/recipes/255756-salat-acecili-s-kuricei
Title: Шоколадные вареники с творогом и вишней, URL: https://food.ru/recipes/257266-shokoladnye-vareniki-s-tvorogom-i-vishnei
Title: Простой блинный торт с черникой, URL: https://food.ru/recipes/256678-blinchiki-s-chernikoi
Title: Зеленый салат с семенами и орехами и м

In [12]:
df.head()

,title,url,recipe,time,count,ingredients,nutrients,allergy,category
0,Гнезда с тефтелями в томатном соусе,https://food.ru/recipes/257868-gnezda-s-teftel...,Крупно натрите морковь. Нарежьте лук кубиками ...,60,4.0,"[{'name': 'Фарш из свинины и говядины', 'qty':...","{'Калории': '163,32', 'Белки': '6,66', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глютен",вторые блюда
1,Тушеные баклажаны с фасолью,https://food.ru/recipes/257915-baklazhany-s-fa...,Секция с рецептом не найдена,0,NaN,[],{},Аллергии не найдены,вторые блюда
2,Cалат из зеленого горошка с сырокопченым карпаччо,https://food.ru/recipes/258025-calat-iz-zeleno...,Секция с рецептом не найдена,0,NaN,[],{},Аллергии не найдены,вторые блюда
3,Легкий салат с кукурузой,https://food.ru/recipes/255758-legkii-salat-s-...,"Положите яйца в кастрюлю, залейте их водой. По...",25,4.0,"[{'name': 'Салат–латук', 'qty': '1', 'grams': ...","{'Калории': '64,5', 'Белки': '3,69', 'Жиры': '...","Белок коровьего молока, Яйцо",вторые блюда
4,Фетучини карбонара,https://food.ru/recipes/257663-fetuchini-karbo...,Отварите фетучини в кипящей подсоленной воде д...,20,4.0,"[{'name': 'Паста фетучини', 'qty': '400', 'gra...","{'Калории': '410,84', 'Белки': '20,5', 'Жиры':...","Белок коровьего молока, Злаки, содержащие глют...",вторые блюда


In [13]:
df.to_csv("recipes_foodru.csv", encoding="utf-8")